# Smoked-fish remaining-time dataset

One run of **4 fish**, two log files (NEPA cut in the middle):

- `fish_data_4`
- `fish_data_4_part2`

Raw columns: `Timestamp, DHT11_Temp_C, DHT11_Humidity_pct, MAX6675_OvenTemp_C, MQ6_ADC, MQ6_Ratio, Weight_g`

**Done** = when you pulled the fish (~13:48), not a guessed plateau.
Time, heat-hours, and remaining minutes count only while the smoker is **on**. The blackout is a pause.

## Load-cell reset (part 2)

Part 2 grams are a new zero. Keep the **change**, pin the first fish-on reading to the last good part-1 weight:

`weight_corrected = W_last - (W_p2_0 - weight_raw)`

Example: last = 1680, part2 starts 1850, later 1760 → `1680 - (1850-1760) = 1590`.

## Three live parameters

1. **Remaining active time** — `remaining_min` (train on this). Clock to pull-off, blackout removed.
2. **Weight drop vs start** — this cook needed ~23.5% gone. Near the end: if it is not there yet, keep going; if it is there **and** oven °C·h are enough, you may finish early.
3. **Oven °C·h** — integral of MAX6675 over active hours. More fish (higher start weight) scales needed hours and °C·h by `new_start / this_start`. DHT11 is not used for heat (it stuck at 100 °C).

Run all cells to write `smoking_remaining_time_dataset.csv`.

In [ ]:
from pathlib import Path
import re
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd()
PART1 = ROOT / "fish_data_4"
PART2 = ROOT / "fish_data_4_part2"
OUT_CSV = ROOT / "smoking_remaining_time_dataset.csv"

HEAT_DONE = 0.98
WEIGHT_DONE = 0.98
MIN_OVEN_FOR_ETA = 80.0

COLS = [
    "timestamp", "dht11_temp_c", "dht11_humidity_pct", "oven_temp_c",
    "mq6_adc", "mq6_ratio", "weight_g",
]
LINE_RE = re.compile(
    r"(\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}),"
    r"([-\d.]+),([-\d.]+),([-\d.]+),([-\d.]+),([-\d.]+),([-\d.]+)"
)


def load_log(path: Path) -> pd.DataFrame:
    rows = []
    text = path.read_text(encoding="utf-8", errors="replace")
    for line in text.splitlines():
        m = LINE_RE.search(line)
        if not m:
            continue
        ts = datetime.strptime(m.group(1), "%Y-%m-%d %H:%M:%S")
        vals = [float(x) for x in m.groups()[1:]]
        rows.append((ts, *vals))
    df = pd.DataFrame(rows, columns=COLS)
    return (
        df.drop_duplicates(subset=["timestamp"])
        .sort_values("timestamp")
        .reset_index(drop=True)
    )


p1 = load_log(PART1)
p2 = load_log(PART2)
print(f"part1: {len(p1)}  {p1.timestamp.min()} → {p1.timestamp.max()}")
print(f"part2: {len(p2)}  {p2.timestamp.min()} → {p2.timestamp.max()}")

## Clean dropouts and smoking windows

Fill 1-sample empty-tray zeros. Keep part 1 until the last still-hot sample. Part 2 starts when the four fish sit on the tray and heat begins; stop at the last stable sample before you unloaded.

In [ ]:
def fill_weight_dropouts(w: np.ndarray, min_fish_g: float = 400.0) -> np.ndarray:
    out = w.copy()
    last = np.nan
    for i, v in enumerate(out):
        if v >= min_fish_g:
            last = v
        elif not np.isnan(last):
            nxt = any(out[j] >= min_fish_g for j in range(i + 1, min(i + 6, len(out))))
            if nxt:
                out[i] = last
    return out


p1 = p1.copy()
p2 = p2.copy()
p1["weight_filled_g"] = fill_weight_dropouts(p1.weight_g.to_numpy())
p2["weight_filled_g"] = fill_weight_dropouts(p2.weight_g.to_numpy())

hot = p1[(p1.oven_temp_c > 370) & (p1.weight_filled_g > 400)]
last_known_row = hot.iloc[-1]
W_LAST = float(last_known_row.weight_filled_g)
print("W_LAST", last_known_row.timestamp, W_LAST, "oven", last_known_row.oven_temp_c)

p1_smoke = p1[p1.timestamp <= last_known_row.timestamp].copy()
FISH_ON = pd.Timestamp("2026-08-25 09:31:04")
DONE = pd.Timestamp("2026-08-25 13:48:34")
p2_smoke = p2[
    (p2.timestamp >= FISH_ON) & (p2.timestamp <= DONE) & (p2.weight_filled_g > 400)
].copy()

W_P2_0 = float(p2_smoke.weight_filled_g.iloc[0])
print("W_P2_0", W_P2_0, "offset", W_LAST - W_P2_0)

p1_smoke["session"] = "part1"
p2_smoke["session"] = "part2"
p1_smoke["weight_corrected_g"] = p1_smoke.weight_filled_g
p2_smoke["weight_corrected_g"] = W_LAST - (W_P2_0 - p2_smoke.weight_filled_g)

## Re-anchor later tray / load-cell steps

If displayed weight jumps by a large step and stays, treat it like the outage reset: keep mass continuous.

In [ ]:
def reanchor_steps(weights: np.ndarray, jump_g: float = 80.0, hold: int = 4) -> np.ndarray:
    w = weights.copy()
    offset = 0.0
    out = np.empty_like(w)
    out[0] = w[0]
    for i in range(1, len(w)):
        dw = (w[i] + offset) - out[i - 1]
        if abs(dw) >= jump_g and i + hold < len(w):
            future = w[i : i + hold]
            if np.max(np.abs(np.diff(future))) < jump_g / 2:
                offset = out[i - 1] - w[i]
        out[i] = w[i] + offset
    return out


p1_smoke["weight_corrected_g"] = reanchor_steps(p1_smoke.weight_corrected_g.to_numpy())
p2_smoke["weight_corrected_g"] = reanchor_steps(p2_smoke.weight_corrected_g.to_numpy())

combo = pd.concat([p1_smoke, p2_smoke], ignore_index=True).sort_values("timestamp").reset_index(drop=True)

dt = combo.timestamp.diff().dt.total_seconds().fillna(0).clip(lower=0, upper=30)
combo["elapsed_smoking_min"] = dt.cumsum() / 60.0
t_end = float(combo.elapsed_smoking_min.iloc[-1])
combo["remaining_min"] = t_end - combo.elapsed_smoking_min

w0 = float(combo.weight_corrected_g.iloc[0])
combo["start_weight_g"] = w0
combo["weight_smooth_g"] = combo.weight_corrected_g.rolling(12, min_periods=1, center=True).median()
combo["weight_loss_from_start_g"] = w0 - combo.weight_corrected_g
combo["moisture_removed_frac"] = combo.weight_loss_from_start_g / w0
combo["drying_rate_g_per_min"] = (
    combo.weight_smooth_g.diff() / combo.elapsed_smoking_min.diff().replace(0, np.nan)
).rolling(24, min_periods=4).median()

# Oven heat dose while active (blackout not counted)
combo["oven_deg_h"] = (combo.oven_temp_c * (dt / 3600.0)).cumsum()
target_deg_h = float(combo.oven_deg_h.iloc[-1])
target_loss_frac = float(combo.moisture_removed_frac.iloc[-1])
target_hours = t_end / 60.0
combo["target_oven_deg_h"] = target_deg_h
combo["target_loss_frac"] = target_loss_frac
combo["target_hours"] = target_hours
combo["heat_progress_frac"] = combo.oven_deg_h / target_deg_h
combo["weight_progress_frac"] = combo.moisture_removed_frac / target_loss_frac

leftover_deg_h = (target_deg_h - combo.oven_deg_h).clip(lower=0)
combo["remaining_min_from_heat"] = leftover_deg_h / combo.oven_temp_c.clip(lower=MIN_OVEN_FOR_ETA) * 60.0

loss_rate = (-combo.drying_rate_g_per_min).clip(lower=0.05)
leftover_g = (target_loss_frac * w0 - combo.weight_loss_from_start_g).clip(lower=0)
combo["remaining_min_from_weight"] = leftover_g / loss_rate
combo.loc[combo.drying_rate_g_per_min.isna(), "remaining_min_from_weight"] = combo.remaining_min

# Live rule: finish early only if heat AND weight targets are met; otherwise keep the longer of clock / heat / weight
both_ok = (combo.heat_progress_frac >= HEAT_DONE) & (combo.weight_progress_frac >= WEIGHT_DONE)
combo["remaining_min_gated"] = np.where(
    both_ok,
    0.0,
    np.maximum.reduce(
        [
            combo.remaining_min.to_numpy(),
            combo.remaining_min_from_heat.to_numpy(),
            combo.remaining_min_from_weight.fillna(combo.remaining_min).to_numpy(),
        ]
    ),
)

print(f"Active smoking {t_end:.1f} min ({target_hours:.2f} h)")
print(f"Start {w0:.1f} g → end {combo.weight_corrected_g.iloc[-1]:.1f} g  loss {target_loss_frac*100:.1f}%")
print(f"Oven dose {target_deg_h:.0f} °C·h  mean oven {combo.oven_temp_c.mean():.0f} °C")
print(f"Join last part1 {p1_smoke.weight_corrected_g.iloc[-1]:.1f}  first part2 raw {p2_smoke.weight_g.iloc[0]:.1f} → {p2_smoke.weight_corrected_g.iloc[0]:.1f}")
print(f"Later loads: scale hours and °C·h by (new_start_g / {w0:.1f})")

In [ ]:
dataset = combo[
    [
        "timestamp",
        "session",
        "elapsed_smoking_min",
        "remaining_min",
        "remaining_min_from_heat",
        "remaining_min_from_weight",
        "remaining_min_gated",
        "start_weight_g",
        "dht11_temp_c",
        "dht11_humidity_pct",
        "oven_temp_c",
        "mq6_adc",
        "mq6_ratio",
        "weight_g",
        "weight_corrected_g",
        "weight_smooth_g",
        "weight_loss_from_start_g",
        "moisture_removed_frac",
        "target_loss_frac",
        "weight_progress_frac",
        "drying_rate_g_per_min",
        "oven_deg_h",
        "target_oven_deg_h",
        "heat_progress_frac",
        "target_hours",
    ]
].copy()
dataset.to_csv(OUT_CSV, index=False)
print("Wrote", OUT_CSV, "rows=", len(dataset))
dataset.describe().T

## Trends

Corrected weight should not jump at the join. `remaining_min` is the training target. Heat and weight progress are the two checks near the end.

In [ ]:
join_x = combo.loc[combo.session.eq("part1"), "elapsed_smoking_min"].iloc[-1]
fig, axes = plt.subplots(4, 1, figsize=(12, 14), sharex=True)

axes[0].plot(combo.elapsed_smoking_min, combo.weight_g, alpha=0.35, label="raw")
axes[0].plot(combo.elapsed_smoking_min, combo.weight_corrected_g, label="corrected")
axes[0].plot(combo.elapsed_smoking_min, combo.weight_smooth_g, label="smoothed")
axes[0].axvline(join_x, color="k", ls="--", lw=1, label="NEPA join")
axes[0].set_ylabel("weight (g)")
axes[0].legend(loc="upper right")

axes[1].plot(combo.elapsed_smoking_min, combo.oven_temp_c, label="oven")
axes[1].plot(combo.elapsed_smoking_min, combo.oven_deg_h, label="°C·h (right scale)")
axes[1].set_ylabel("oven °C / °C·h")
axes[1].legend()

axes[2].plot(combo.elapsed_smoking_min, combo.heat_progress_frac, label="heat progress")
axes[2].plot(combo.elapsed_smoking_min, combo.weight_progress_frac, label="weight-loss progress")
axes[2].axhline(1.0, color="k", lw=0.8)
axes[2].set_ylabel("fraction of this cook's total")
axes[2].legend()

axes[3].plot(combo.elapsed_smoking_min, combo.remaining_min, label="remaining (to pull)")
axes[3].plot(combo.elapsed_smoking_min, combo.remaining_min_gated, alpha=0.8, label="gated (early / extend)")
axes[3].set_ylabel("min")
axes[3].set_xlabel("elapsed active smoking (min)")
axes[3].legend()

plt.tight_layout()
plt.show()

## Optional model check

Train on the three ideas: oven temp + heat so far, weight loss vs start, remaining time as the label.
Time-ordered split (first 70% / last 30%). Skip this cell if `sklearn` is not installed.
Do not ship `elapsed_smoking_min` as a feature if the device must work without knowing total duration.

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

feat = [
    "oven_temp_c",
    "oven_deg_h",
    "heat_progress_frac",
    "weight_corrected_g",
    "weight_loss_from_start_g",
    "moisture_removed_frac",
    "weight_progress_frac",
    "drying_rate_g_per_min",
]
Xy = dataset.dropna(subset=feat + ["remaining_min"])
cut = int(len(Xy) * 0.7)
X_train, X_test = Xy[feat].iloc[:cut], Xy[feat].iloc[cut:]
y_train, y_test = Xy.remaining_min.iloc[:cut], Xy.remaining_min.iloc[cut:]

model = RandomForestRegressor(n_estimators=200, min_samples_leaf=8, random_state=0, n_jobs=-1)
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f"MAE {mean_absolute_error(y_test, pred):.2f} min   R2 {r2_score(y_test, pred):.3f}")
print(pd.Series(model.feature_importances_, index=feat).sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(Xy.elapsed_smoking_min.iloc[cut:], y_test.values, label="true remaining")
ax.plot(Xy.elapsed_smoking_min.iloc[cut:], pred, label="RF")
ax.set_xlabel("elapsed smoking min")
ax.set_ylabel("remaining min")
ax.legend()
plt.tight_layout()
plt.show()